In [ ]:
MODELS = {
    "gemma3:1b":    "google/gemma-3-1b-it",
    "qwen3:1.7b":   "Qwen/Qwen3-1.7B",
    "qwen3.5:0.8b": "Qwen/Qwen3.5-0.8B",
}
OLLAMA_BASE     = "http://localhost:11434"
EVAL_SAMPLES    = 300
SEED            = 42
NUM_CTX         = 4096
MAX_TEXT_TOKENS = 3000
TEMPERATURE     = 0.3
REPEAT_PENALTY  = 1.15
RESULTS_PATH    = "../tmp/model_selection.json"

In [2]:
import json
import random
import requests
from pathlib import Path

import matplotlib.pyplot as plt
from datasets import load_dataset
from litellm import completion
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

from safesum.utils.data import truncate_to_tokens
from safesum.metrics import MRougeScorer, Score
from safesum.utils import make_uk_sentence_splitter, make_uk_tokenizer

console = Console()

In [3]:
scorer = MRougeScorer(
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
    tokenizer=make_uk_tokenizer(),
    sentence_splitter=make_uk_sentence_splitter(),
)

In [4]:
ds = load_dataset("csebuetnlp/xlsum", "ukrainian", trust_remote_code=True)
random.seed(SEED)
val_indices = random.sample(range(len(ds["validation"])), EVAL_SAMPLES)
val_sample  = ds["validation"].select(val_indices)
console.print(f"Dataset loaded — using [bold]{len(val_sample)}[/bold] validation samples")

Dataset loaded — using 300 validation samples

In [5]:
def generate(model_key: str, messages: list, think: bool = False) -> str:
    resp = completion(
        model=f"ollama/{model_key}",
        messages=messages,
        api_base=OLLAMA_BASE,
        think=think,
        temperature=TEMPERATURE,
        extra_body={"options": {"num_ctx": NUM_CTX, "repeat_penalty": REPEAT_PENALTY}},
    )
    return resp.choices[0].message.content.strip()


def unload_model(model_key: str) -> None:
    """Evict model from ollama VRAM/RAM immediately (keep_alive=0)."""
    requests.post(
        f"{OLLAMA_BASE}/api/generate",
        json={"model": model_key, "keep_alive": 0},
        timeout=30,
    )


def truncate_to_tokens(text: str, tokenizer: AutoTokenizer, max_tokens: int) -> str:
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)


# ── Prompts ─────────────────────────────────────────────────────────────────

def make_uk_question_messages(question: str) -> list:
    system_prompt = "Ти — корисний помічник. Відповідай виключно українською мовою."

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]


def make_summary_messages(title: str, text: str) -> list:
    system_prompt = (
        "Ти — помічник для створення коротких підсумків українських новин. "
        "Відповідай виключно українською мовою."
    )

    return [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Напиши підсумок рівно 1–2 реченнями. Не додавай нічого зайвого.\n\n"
                f"Заголовок: {title}\n\n"
                f"Текст: {text}\n\n"
                "Підсумок:"
            ),
        },
    ]


# ── Results persistence ──────────────────────────────────────────────────────

def save_results(results: dict, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    serializable = {
        model_key: [
            {
                "title":  r["title"],
                "text":   r["text"],
                "ref":    r["ref"],
                "pred":   r["pred"],
                "scores": {
                    k: {"precision": v.precision, "recall": v.recall, "fmeasure": v.fmeasure}
                    for k, v in r["scores"].items()
                },
            }
            for r in data
        ]
        for model_key, data in results.items()
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)


def load_results(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    return {
        model_key: [
            {
                **r,
                "scores": {
                    k: Score(precision=v["precision"], recall=v["recall"], fmeasure=v["fmeasure"])
                    for k, v in r["scores"].items()
                },
            }
            for r in data
        ]
        for model_key, data in raw.items()
    }

## 1. Ukraine knowledge test

In [6]:
def ukraine_knowledge_test(test_name: str, question: str):
    console.rule(f"[bold blue]Тест: {test_name}[/bold blue]")
    for model_key, hf_name in MODELS.items():
        answer = generate(model_key, make_uk_question_messages(question))
        console.print(
            Panel(
                answer,
                title=f"[bold green]{hf_name}[/bold green]  [dim]({model_key})[/dim]",
                border_style="blue",
                padding=(1, 2),
            )
        )

In [7]:
ukraine_knowledge_test(
    test_name="basic_knowledge",
    question="Розкажи про Україну",
)

────────────────────────────────────────────── Тест: basic_knowledge ──────────────────────────────────────────────

╭─────────────────────────────────────── google/gemma-3-1b-it  (gemma3:1b) ───────────────────────────────────────╮
│                                                                                                                 │
│  Звісно! Україна – це прекрасна країна з багатишою історією та культурою. Ось короткий огляд:                   │
│                                                                                                                 │
│  **Географія та природа:**                                                                                      │
│                                                                                                                 │
│  *   **Розташування:** Україна розташована в Східноєвропейській гірській півкулі, на переході між Балканами та  │
│  Кавказом.                                                                                                      │
│  *   **Рельєф:**  У країні різноманітний рельєф – від гір (Карпати, Крихта, Бордюки) до рівнин (Волік, Донбас)  │
│  та узбережжя морів (Чорне, Азовське, Керченське).                                                              │
│  *   **Природні ресурси:** Україна має великі запаси води, лісів, пасовищ, а також багатство природних          │
│  корисних копалин.                                                                                              │
│  *   **Погода:**  Зазвичай, клімат помірний, з сприятливою зимою та теплою, су हिरкою літом.                     │
│                                                                                                                 │
│  **Історія:**                                                                                                   │
│                                                                                                                 │
│  *   **Древні часи:** Джерелом є давніна –  історії заснування України, в якій існували різні племена та        │
│  цивілізації.                                                                                                   │
│  *   **Завоювання та вплив:** Україна була частиною різних держав, включаючи Київську Русь, Половець, Велику    │
│  Моравию, Галійську та Османську імперію.                                                                       │
│  *   **Захист від Золотої Орди:** У 13 столітті Україна захищалася від Золотої Орди, що значно вплинуло на її   │
│  розвиток.                                                                                                      │
│  *   **Польний війна:**  У 15 столітті відбулася Польнийська війна, яка перетворила Україну на «срібну жипу»    │
│  для Російської імперії.                                                                                        │
│  *   **Незалежність:** 1917 рік -  заснування України як незалежної держави.                                    │
│                                                                                                                 │
│  **Культура:**                                                                                                  │
│                                                                                                                 │
│  *   **Мова:** Українська мова –  один з найменш задіяних в світі мов.                                          │
│  *   **Православ'я:**  Україна є найбільшим земстям православ'я в Європі.                                       │
│  *   **Традиції:**  Українська культура відома своїм народним танцями, народним мистецтвом,  віруваннями та     │
│  обрядами.                                                                                                      │
│  *   **Музика та література:**  Українська музика,  тексти, і до творчості відомі  автори, такі як Тарас        │
│  Шевченко, Іван Франко, Леся Українка.                                                                          │
│  *   **Їжа:**  Українська кухня дуже різноманітна,  з

╭───────────────────────────────────────── Qwen/Qwen3-1.7B  (qwen3:1.7b) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Україна — це країна, що належить до Європейського півдня, є частиною Європейського континенту. Вона має        │
│  високу культуру, традиції та історичну значущість. Україна відома своєю музичною, літературною та публічною    │
│  культурою, а також своєю державною історією. Вона має багато різних націн, в тому числі українців, русів,      │
│  молдовинців, сербів, та інших. Україна має багато природних заповідів, міста, річок, гор і іншого природного   │
│  багатостійкого суспінду. Вона є частиною Європейського союзу, а також участь у багатьох міжнародних проектах.  │
│  Україна має багато різних мов, в тому числі українську, руску, молдовську та інші. Вона є частиною багатьох    │
│  міжнародних організацій, таких як ООН, ЄС, Європейський союз та інші. Україна має багато різних традицій, в    │
│  тому числі традиції від усіх епох, від давніх та до сьогодні. Вона є частиною багатьох міжнародних             │
│  організацій, таких як ООН, ЄС, Європейський союз та інші. Україна має багато різних мов, в тому числі          │
│  українську, руску, молдовську та інші. Вона є частиною багатьох міжнародних організацій, таких як ООН, ЄС,     │
│  Європейський союз та інші. Україна має багато різних традицій, в тому числі традиції від усіх епох, від        │
│  давніх та до сьогодні.                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Qwen/Qwen3.5-0.8B  (qwen3.5:0.8b) ───────────────────────────────────────╮
│                                                                                                                 │
│  Усі українці – це єдина частина історії України, історії України на 500-літню сторінку.                        │
│                                                                                                                 │
│  **Назва держави**                                                                                              │
│  За назвою «Україна» називають країну, де виступають українці. Вважається, що це історична основа назви         │
│  «Українське», яке означає «українські».                                                                        │
│                                                                                                                 │
│  **Назва країни**                                                                                               │
│  Прямо від назви країни, що означає «князівка», що є корінням.                                                  │
│                                                                                                                 │
│  **Особливості**                                                                                                │
│  Усі українці – це частина історії.                                                                             │
│                                                                                                                 │
│  **Відомі дати**                                                                                                │
│  У 1701 році була заснована імперія. У 1917 році почалася революція.                                            │
│                                                                                                                 │
│  **Кількість людей**                                                                                            │
│  Існує близько 180 мільярдів українців. Усе це єдина частина історії України.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
ukraine_knowledge_test(
    test_name="war_knowledge",
    question="Розкажи про війну в Україні",
)

─────────────────────────────────────────────── Тест: war_knowledge ───────────────────────────────────────────────

╭─────────────────────────────────────── google/gemma-3-1b-it  (gemma3:1b) ───────────────────────────────────────╮
│                                                                                                                 │
│  Добре, я розкажу вам про війну в Україні. Це складний і трагічний конфлікт, який зараз триває вже більше 7     │
│  років. Ось основні моменти, які важливі:                                                                       │
│                                                                                                                 │
│  **Що відбувається?**                                                                                           │
│                                                                                                                 │
│  Війна розпочалася у 2014 році, коли Росія розпочала незаконну агресію на територію України. З того часу        │
│  конфлікт перетворився на масштабну війну між Україною та Росією.                                               │
│                                                                                                                 │
│  **Основні точки конфлікту:**                                                                                   │
│                                                                                                                 │
│  *   **Зовнішня блокада:** Росія блокує доступ до українських портів, що значно впливає на імпорт необхідних    │
│  для виробництва товарів, включаючи зерно, харчові продукти та інші.                                            │
│  *   **Поглиблення конфлікту:** Почалися активні бойові дії в різних регіонах України, зокрема на Донбасі –     │
│  регіоні, де тривають бойові дії між українськими військовими та російськими військами.                         │
│  *   **Посилення російських ударів:** Росія проводить постійні повітряні удари по українських містах,           │
│  включаючи енергетичну інфраструктуру, що призводить до знецінення населення.                                   │
│  *   **Обстріли зброєю:** Зростає кількість обстрілів українських міст зброєю, зокрема з використанням          │
│  артилерії, мінометрів та інших видів озброєння. Це призводить до руйнувань і жертв серед цивільних.            │
│  *   **Міжнародна підтримка:** Україна отримує підтримку від багатьох країн світу, зокрема від ЄС, США,         │
│  Великобританії, Канади, країн Африки, Азії та інших.                                                           │
│                                                                                                                 │
│  **Географія конфлікту:**                                                                                       │
│                                                                                                                 │
│  Війна розгортається в різних регіонах України:                                                                 │
│                                                                                                                 │
│  *   **Донецьк та Луганськ:** Це центри бойових дій, де руйнуються міста і села.                                │
│  *   **Дніпроск:**  Зростає ризик виходу на берег Дніпра.                                                       │
│  *   **Дніпровська область:**  Постійно відбуваються бої.                                                       │
│  *   **Інші території:**  Включають регіони, де російські війська здійснюють наплив.                            │
│                                                                                                                 │
│  **Етичні та гуманітарні наслідки:**                                                                            │
│                                                                                                                 │
│  *   **Знищення цивільного населення:**  Війна призвел

╭───────────────────────────────────────── Qwen/Qwen3-1.7B  (qwen3:1.7b) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Війна в Україні — це воєнна ситуація, яка відбувся з 24 лютого 2022 року. Вона відбулася після того, як        │
│  Україна відмовилася від узгодженого пакету з Україною (УПУ) відповідно до Договіру про зупинути війну. Війна   │
│  відбулася між Україною та Росією, що відбувся на території України. Це війна, яка відбулася в рамках змінених  │
│  умов, що відбувся після відмови України в узгодженому пакеті з Україною. Війна відбулася у відкритому          │
│  просторі, без військового відмовлення. Війна відбулася з 24 лютого 2022 року, і вона відбулася в рамках        │
│  змінених умов, що відбувся після відмови України в узгодженому пакеті з Україною. Війна відбулася відповідно   │
│  до Договіру про зупинути війну.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Qwen/Qwen3.5-0.8B  (qwen3.5:0.8b) ───────────────────────────────────────╮
│                                                                                                                 │
│  На сьогоднішня війна (2022) на сході України стала інтенсивною битвою між Українською армією та російськими    │
│  арміями. Вона розглядалася як найбільш важливе перемоги у галузі воєнних дій, що спочатку наразі неможливе     │
│  для Російської Федерації, а також як критична умова для продовження цієї конфлікти.                            │
│                                                                                                                 │
│  Нижче наведено ключові аспекти цього етапу:                                                                    │
│                                                                                                                 │
│  ### 1. Битви і стратегічні події                                                                               │
│  *   **Сіверська битва 2022 року:** Це найважливіша операція у сучасних війнах. Український контингент          │
│  розгортався в районі Білостою, де він перетнув кордон із Російською Федерацією. Вона розгромила 11-й корпус    │
│  Російських військ і прибуло на південний сході країни.                                                         │
│  *   **Ліпенська битва:** Після перемоги біля Ліпи, Україна змусила Росію повернутися на південний заход        │
│  України, щоб відновити контроль над частинами.                                                                 │
│  *   **Львівська битва:** Вона стала першою після 1990-х років битвою, де російські військи були розбиті і      │
│  повернули контроль над територією, розділеним від України (в тому числі місто Львів і його окрузі).            │
│                                                                                                                 │
│  ### 2. Роль України як переможець                                                                              │
│  Виключно український воєнний флот став першим воїном світу, що вийшов на кордон з Російською Федерацією. Вона  │
│  зберігла величезний запас танків, зброї, військової техніки та муніфекту. Це значно обмежило можливість        │
│  російських військ втратити свій флот на суходолі.                                                              │
│                                                                                                                 │
│  ### 3. Тимчасовий перерв                                                                                       │
│  Після битви на сході Україна зробила перемогу, але війна продовжувалася до кінця 2022 року в інших регіонах    │
│  (запорозьки, південні регіони). Тимчасовий перерв від 3 до 5 квітня 2023 року зумовлено тим, що російські      │
│  війська були розбиті на сході, а в західних регіонах України зменшилися частини збройних сили.                 │
│                                                                                                                 │
│  ### 4. Вплив на мир та майбутнє                                                                                │
│  Вінциновий термін (з 1 жовтня 2022 року до кінця 2022 року) був найкоротшим у історії. Він зумів спостерігати  │
│  за ситуацією, яка стала складнішою після наші наразі. У 2023 році на час зумовленому перервами українська      │
│  сторона стала переможецями, змінюючи термін міжнародної зони свободи, зокрема у випадку на території           │
│  Російської Федерації.                                                                                          │
│                                                                                                                 │
│  ### 5. Ресурси та майбутнє                                                                                     │
│  У цей період українська сторона постійно розробляла с

## 2. Summarisation evaluation

In [9]:
results_path = Path(RESULTS_PATH)

if results_path.exists():
    results = load_results(RESULTS_PATH)
    console.print(f"[dim]Loaded cached results from [bold]{RESULTS_PATH}[/bold] — skipping generation[/dim]")
else:
    results: dict[str, list] = {}

    for model_key, hf_name in MODELS.items():
        console.print(f"\n[bold]Evaluating [green]{hf_name}[/green] ({model_key})...[/bold]")

        tokenizer    = AutoTokenizer.from_pretrained(hf_name)
        model_results = []

        for row in tqdm(val_sample, desc=model_key, leave=True):
            text   = truncate_to_tokens(row["text"], tokenizer, MAX_TEXT_TOKENS)
            pred   = generate(model_key, make_summary_messages(row["title"], text))
            scores = scorer.score(row["summary"], pred)
            model_results.append(
                {
                    "title":  row["title"],
                    "text":   row["text"],
                    "ref":    row["summary"],
                    "pred":   pred,
                    "scores": scores,
                }
            )

        results[model_key] = model_results
        unload_model(model_key)
        console.print(f"[green]✓[/green] {model_key} done and unloaded ({len(model_results)} samples)")

    save_results(results, RESULTS_PATH)
    console.print(f"[green]Results saved to [bold]{RESULTS_PATH}[/bold][/green]")

Evaluating google/gemma-3-1b-it (gemma3:1b)...

gemma3:1b:   0%|          | 0/300 [00:00<?, ?it/s]

✓ gemma3:1b done and unloaded (300 samples)

Evaluating Qwen/Qwen3-1.7B (qwen3:1.7b)...

qwen3:1.7b:   0%|          | 0/300 [00:00<?, ?it/s]

✓ qwen3:1.7b done and unloaded (300 samples)

Evaluating Qwen/Qwen3.5-0.8B (qwen3.5:0.8b)...

qwen3.5:0.8b:   0%|          | 0/300 [00:00<?, ?it/s]

✓ qwen3.5:0.8b done and unloaded (300 samples)

Results saved to results/model_selection.json

## 3. ROUGE score comparison

In [10]:
ROUGE_TYPES = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

table = Table(show_header=True, header_style="bold magenta", title="ROUGE Scores — macro-averaged F1 (%)")
table.add_column("Model",    style="cyan",  no_wrap=True)
table.add_column("HF name",  style="dim")
for rt in ROUGE_TYPES:
    table.add_column(rt, justify="right")

for model_key, hf_name in MODELS.items():
    data = results[model_key]
    avg  = scorer.score_corpus(
        [r["ref"]  for r in data],
        [r["pred"] for r in data],
    )
    table.add_row(
        model_key,
        hf_name,
        *[f"{avg[rt].fmeasure * 100:.1f}" for rt in ROUGE_TYPES],
    )

console.print(table)

                     ROUGE Scores — macro-averaged F1 (%)                     
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┓
┃ Model        ┃ HF name              ┃ rouge1 ┃ rouge2 ┃ rougeL ┃ rougeLsum ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━┩
│ gemma3:1b    │ google/gemma-3-1b-it │   15.0 │    3.4 │   11.6 │      12.8 │
│ qwen3:1.7b   │ Qwen/Qwen3-1.7B      │   17.0 │    4.2 │   13.7 │      14.7 │
│ qwen3.5:0.8b │ Qwen/Qwen3.5-0.8B    │   12.4 │    2.1 │    9.2 │      10.3 │
└──────────────┴──────────────────────┴────────┴────────┴────────┴───────────┘

## 4. Best & worst summary examples

In [12]:
def _fmt_body(title: str, text: str, ref: str, pred: str, scores: dict) -> str:
    r1   = scores["rouge1"].fmeasure * 100
    r2   = scores["rouge2"].fmeasure * 100
    rl   = scores["rougeL"].fmeasure * 100
    rlsum = scores["rougeLsum"].fmeasure * 100
    snippet = text[:300] + ("..." if len(text) > 300 else "")
    return (
        f"[bold]Заголовок:[/bold] {title}\n\n"
        f"[bold]Текст (уривок):[/bold] {snippet}\n\n"
        f"[bold]Референс:[/bold] {ref}\n\n"
        f"[bold]Прогноз:[/bold]   {pred}\n\n"
        f"[dim]rouge1={r1:.1f}  rouge2={r2:.1f}  rougeL={rl:.1f}  rougeLsum={rlsum:.1f}[/dim]"
    )


console.rule("[bold blue]Приклади підсумків — найкращий та найгірший[/bold blue]")

for model_key, hf_name in MODELS.items():
    data       = results[model_key]
    by_lsum    = sorted(data, key=lambda r: r["scores"]["rougeLsum"].fmeasure)
    worst, best = by_lsum[0], by_lsum[-1]

    console.print(f"\n[bold cyan]{hf_name}[/bold cyan]  [dim]({model_key})[/dim]")
    console.print(
        Panel(
            _fmt_body(best["title"], best["text"], best["ref"], best["pred"], best["scores"]),
            title="[bold green]Найкращий підсумок[/bold green]",
            border_style="green",
            padding=(1, 2),
        )
    )
    console.print(
        Panel(
            _fmt_body(worst["title"], worst["text"], worst["ref"], worst["pred"], worst["scores"]),
            title="[bold red]Найгірший підсумок[/bold red]",
            border_style="red",
            padding=(1, 2),
        )
    )

─────────────────────────────────── Приклади підсумків — найкращий та найгірший ───────────────────────────────────

google/gemma-3-1b-it  (gemma3:1b)

╭────────────────────────────────────────────── Найкращий підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Transparency закликала Раду впорядкувати доступ до інформації                                       │
│                                                                                                                 │
│  Текст (уривок): Із відкритим листом до голови парламенту Володимира Рибака з цього приводу звернулася          │
│  регіональний директор організації в Європі і Центральній Азії Ганна Кох. "Україна планує підписати восени      │
│  Угоду про асоціацію з Європейським Союзом і повинна продемонструвати прогрес у низці областей, зокрема в       │
│  сфе...                                                                                                         │
│                                                                                                                 │
│  Референс: Міжнародна організація Transparency International закликала Верховну Раду впорядкувати українське    │
│  законодавство відповідно до раніше ухваленого закону про доступ до публічної інформації.                       │
│                                                                                                                 │
│  Прогноз:   Transparency International закликає Раду України схвалити законопроект 0947, який передбачає        │
│  переформулювання законодавства в сфері доступу до інформації, зокрема, внесення змін до законів про            │
│  адміністративні правопорушення та доступ до публічної інформації.                                              │
│                                                                                                                 │
│  rouge1=36.0  rouge2=16.7  rougeL=36.0  rougeLsum=36.0                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Найгірший підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Нові технології, аби відвернути загрозу літакам                                                     │
│                                                                                                                 │
│  Текст (уривок): Але серпень 2006 запам'ятається тим, що вперше повністю заборонено проносити ручний багаж на   │
│  борт літаків. Фахівці кажуть, що цей крок став "драконівським", у порівнянні з найжорсткішими заходами         │
│  безпеки, що до них вдалися авіалінії по всьому світі. Пасажирам у Лондоні від 10 серпня дозволяється про...    │
│                                                                                                                 │
│  Референс: Після нападів 11 вересня авіаперевезення змінилися назавжди.                                         │
│                                                                                                                 │
│  Прогноз:   У Британії запроваджена заборона на ручний багаж на літаках, що спрямована на запобігання проносу   │
│  на борт вибухових речовин та годинникових механізмів. Для запобігання цьому, запроваджено заборону на напої    │
│  та гелеві речовини.                                                                                            │
│                                                                                                                 │
│  rouge1=0.0  rouge2=0.0  rougeL=0.0  rougeLsum=0.0                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Qwen/Qwen3-1.7B  (qwen3:1.7b)

╭────────────────────────────────────────────── Найкращий підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Обвал у нічному клубі на Тенерифе: 22 постраждалих                                                  │
│                                                                                                                 │
│  Текст (уривок): Відвідувачі клубу через діру, яка утворилася, впали з висоти одного поверху до                 │
│  напівпідвального пабу. Інцидент стався о 02:30 за місцевим часом (00:30 за Києвом) у ніч на неділю. Гей-клуб,  │
│  в якому стався обвал, знаходиться на популярному серед туристів курорті Плая-де-лас-Америкас в західній        │
│  частині...                                                                                                     │
│                                                                                                                 │
│  Референс: У нічному клубі на іспанському острові Тенерифе завалилася танцювальна зала, внаслідок інциденту 22  │
│  людини отримали травми.                                                                                        │
│                                                                                                                 │
│  Прогноз:   Обвал у нічному клубі на Тенерифе звідповідно призвів до 22 постраждалих, з яких двоє отримали      │
│  переломи стегнової кістки.                                                                                     │
│                                                                                                                 │
│  rouge1=41.2  rouge2=18.8  rougeL=41.2  rougeLsum=41.2                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Найгірший підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Нові технології, аби відвернути загрозу літакам                                                     │
│                                                                                                                 │
│  Текст (уривок): Але серпень 2006 запам'ятається тим, що вперше повністю заборонено проносити ручний багаж на   │
│  борт літаків. Фахівці кажуть, що цей крок став "драконівським", у порівнянні з найжорсткішими заходами         │
│  безпеки, що до них вдалися авіалінії по всьому світі. Пасажирам у Лондоні від 10 серпня дозволяється про...    │
│                                                                                                                 │
│  Референс: Після нападів 11 вересня авіаперевезення змінилися назавжди.                                         │
│                                                                                                                 │
│  Прогноз:   Нові технології змінені заходи безпеки літаків, щоб відвернути загрозу. Заборона ручного багажу і   │
│  контроль над напоєм і гелевою речовиною є ключовим кроком для підбиття безпеки.                                │
│                                                                                                                 │
│  rouge1=0.0  rouge2=0.0  rougeL=0.0  rougeLsum=0.0                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Qwen/Qwen3.5-0.8B  (qwen3.5:0.8b)

╭────────────────────────────────────────────── Найкращий підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Спецслужби Туреччини: в саудівському консульстві вбили і розчленували журналіста                    │
│                                                                                                                 │
│  Текст (уривок): Відтоді як журналіст зайшов до будівлі дипмісії, його більше ніхто не бачив. Пан Хашоггі       │
│  критикував саудівську владу, був автором матеріалів у газеті Washington Post. Він жив у США, а зник після      │
│  того, як увійшов до будівлі консульства в Стамбулі 2 жовтня. Саудівська Аравія відкидає звинувачення і ст...   │
│                                                                                                                 │
│  Референс: У турецької влади є відео- та аудіозаписи, які доводять, що саудівського журналіста Джамаля          │
│  Хашоггі, який зник, убили в консульстві Саудівської Аравії у Стамбулі. Про це повідомило близьке до спецслужб  │
│  Туреччини джерело BBC.                                                                                         │
│                                                                                                                 │
│  Прогноз:   Саудівська консульство в Туреччині вбивла та розчленувала саудівського журналіста Хашоггі, який     │
│  заходились в будівлю дипмісії. Це відбулося через короткий час (до 2 жовтня 2021), що призведе до пошкодження  │
│  репутації саудівського принца-наступника, утиску відносин та загрози з боку Саудівської Аравії.                │
│                                                                                                                 │
│  rouge1=30.6  rouge2=8.6  rougeL=22.2  rougeLsum=30.6                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Найгірший підсумок ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Заголовок: Нові технології, аби відвернути загрозу літакам                                                     │
│                                                                                                                 │
│  Текст (уривок): Але серпень 2006 запам'ятається тим, що вперше повністю заборонено проносити ручний багаж на   │
│  борт літаків. Фахівці кажуть, що цей крок став "драконівським", у порівнянні з найжорсткішими заходами         │
│  безпеки, що до них вдалися авіалінії по всьому світі. Пасажирам у Лондоні від 10 серпня дозволяється про...    │
│                                                                                                                 │
│  Референс: Після нападів 11 вересня авіаперевезення змінилися назавжди.                                         │
│                                                                                                                 │
│  Прогноз:   Експертність країни не зможе гарантувати, що найнижчі технічні рівні автоматизації можуть не        │
│  відсутність вибухових речовин у біоподобній матерії на борт літака, і це не може бути вирішувальним загрозою.  │
│                                                                                                                 │
│  rouge1=0.0  rouge2=0.0  rougeL=0.0  rougeLsum=0.0                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯